# Fine-tune FLAN-T5-small for legal clause simplification

Trains `google/flan-t5-small` to rewrite legal clauses in plain English. Source data: the **Plain English Contracts** corpus from Manor & Li (2019), ~440 hand-aligned clause/simplification pairs.

Output: `freak3123/flan-t5-simplify-v1` on the HF Hub.

**Before you Run All:**
1. Settings → Accelerator → **GPU T4 x2** (or T4).
2. Settings → Internet → **On**.
3. Add-ons → Secrets → ensure `HF_TOKEN` is set (write-scope).
4. Click Run All. Training takes ~12–20 min on T4.

After this finishes, the model is callable from `apps/ml/app/pipeline/rewrite.py` (the integration is already wired).

In [ ]:
!pip install -q --upgrade transformers==4.46.3 datasets==3.1.0 accelerate==1.1.1 sentencepiece==0.2.0 evaluate==0.4.3 rouge-score==0.1.2 huggingface_hub==0.26.2

In [ ]:
import os, json, random, urllib.request
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
from huggingface_hub import login
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'Set HF_TOKEN as a Kaggle secret (Add-ons → Secrets).'
login(token=HF_TOKEN, add_to_git_credential=False)

In [ ]:
BASE_MODEL = 'google/flan-t5-small'
REPO_ID = 'freak3123/flan-t5-simplify-v1'
OUTPUT_DIR = '/kaggle/working/flan-t5-simplify-v1'
MAX_INPUT = 384
MAX_TARGET = 192
BATCH_SIZE = 8
EVAL_BATCH = 16
EPOCHS = 10
LR = 3e-4
PREFIX = 'simplify legal: '

In [ ]:
# Pull the Plain English Contracts dataset from Manor & Li (2019) GitHub mirror.
DATA_PATH = Path('/kaggle/working/plain_english_contracts.json')
if not DATA_PATH.exists():
    url = 'https://raw.githubusercontent.com/lauramanor/legal_summarization/master/all_v1.json'
    urllib.request.urlretrieve(url, DATA_PATH)
with DATA_PATH.open(encoding='utf-8') as f:
    raw = json.load(f)
print(f'top-level type: {type(raw).__name__}; size: {len(raw) if hasattr(raw, "__len__") else "?"}')
print('first key sample:', list(raw.keys())[:3] if isinstance(raw, dict) else raw[:1])

In [ ]:
# Normalise into (legal_text, plain_text) pairs.
# The Manor & Li dataset stores entries with fields 'original_text' and 'reference_summary',
# keyed by document id. We accept a few variants in case the schema changed.
pairs = []
iterable = raw.values() if isinstance(raw, dict) else raw
for ex in iterable:
    if not isinstance(ex, dict):
        continue
    src = ex.get('original_text') or ex.get('text') or ex.get('legal') or ex.get('input')
    tgt = ex.get('reference_summary') or ex.get('summary') or ex.get('plain') or ex.get('output')
    if isinstance(src, list):
        src = ' '.join(src)
    if isinstance(tgt, list):
        tgt = ' '.join(tgt)
    if not src or not tgt:
        continue
    src = src.strip()
    tgt = tgt.strip()
    if len(src) < 30 or len(tgt) < 15:
        continue
    pairs.append((src, tgt))

print(f'usable pairs: {len(pairs)}')
if pairs:
    print('\n--- sample pair ---')
    print('LEGAL :', pairs[0][0][:200])
    print('PLAIN :', pairs[0][1][:200])

In [ ]:
# If the public dataset shape is unexpected, augment with synthetic pairs from
# the project's seed CSV using the project's template rewrites. This guarantees
# at least a few hundred clean training pairs even in a worst case.
from pathlib import Path
import csv

SEED_CSV = next((p for p in [
    Path('/kaggle/input/legal-clauses-seed/clauses_seed.csv'),
    Path('/kaggle/input/clauses-seed/clauses_seed.csv'),
] if p.exists()), None)

def _trivial_simplify(text):
    import re
    out = text
    for pat, rep in [
        (r'\bnotwithstanding\b', 'even though'),
        (r'\bin no event shall\b', 'neither side can'),
        (r'\bshall not\b', 'cannot'),
        (r'\bshall\b', 'must'),
        (r'\bhereinafter\b', 'from now on'),
        (r'\bherein\b', 'in this contract'),
        (r'\bhereby\b', ''),
        (r'\bthereof\b', 'of it'),
        (r'\bpursuant to\b', 'under'),
        (r'\bsubject to\b', 'as long as'),
        (r'\bprovided that\b', 'as long as'),
        (r'\bdue and payable\b', 'due'),
        (r'\bany and all\b', 'all'),
    ]:
        out = re.sub(pat, rep, out, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', out).strip()

if SEED_CSV and len(pairs) < 400:
    print(f'augmenting with seed CSV from {SEED_CSV}')
    with SEED_CSV.open(encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            src = row['text']
            tgt = _trivial_simplify(src)
            if len(tgt) >= 20 and tgt != src:
                pairs.append((src, tgt))
    print(f'total pairs after aug: {len(pairs)}')

random.shuffle(pairs)

In [ ]:
# 90/10 split (small dataset — keep most for training)
n_val = max(40, len(pairs) // 10)
val_pairs = pairs[:n_val]
train_pairs = pairs[n_val:]
print(f'train={len(train_pairs)} val={len(val_pairs)}')

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_features(batch):
    inputs = [PREFIX + s for s in batch['src']]
    model_inputs = tokenizer(inputs, max_length=MAX_INPUT, truncation=True)
    labels = tokenizer(text_target=batch['tgt'], max_length=MAX_TARGET, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_dict({'src': [a for a, _ in train_pairs], 'tgt': [b for _, b in train_pairs]}).map(to_features, batched=True, remove_columns=['src', 'tgt'])
val_ds = Dataset.from_dict({'src': [a for a, _ in val_pairs], 'tgt': [b for _, b in val_pairs]}).map(to_features, batched=True, remove_columns=['src', 'tgt'])

model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
print('loaded base model')

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate

rouge = evaluate.load('rouge')

def metrics_fn(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    if preds.ndim == 3:
        preds = preds.argmax(-1)
    # Replace -100 (ignore index) with pad in BOTH arrays before decoding.
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    preds_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels_text = tokenizer.batch_decode(labels, skip_special_tokens=True)
    out = rouge.compute(predictions=preds_text, references=labels_text, use_stemmer=True)
    return {k: round(float(v), 4) for k, v in out.items()}

args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    fp16=False,
    logging_steps=20,
    report_to='none',
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=metrics_fn,
)

In [ ]:
trainer.train()

In [ ]:
# Spot-check on a few held-out items
model.eval()
device = next(model.parameters()).device
for src, tgt in val_pairs[:5]:
    enc = tokenizer(PREFIX + src, return_tensors='pt', max_length=MAX_INPUT, truncation=True).to(device)
    out = model.generate(**enc, max_length=MAX_TARGET, num_beams=4, early_stopping=True)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print('LEGAL :', src[:160])
    print('REF   :', tgt[:160])
    print('MODEL :', text[:160])
    print('---')

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

from huggingface_hub import HfApi, upload_folder
HfApi().create_repo(repo_id=REPO_ID, exist_ok=True, private=False)
upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id=REPO_ID,
    commit_message=f'FLAN-T5-small fine-tuned for legal clause simplification',
)
print(f'
DONE: model pushed to https://huggingface.co/{REPO_ID}')

## Next steps

Paste back the model URL and the rougeL on val. The project's `apps/ml/app/pipeline/rewrite.py` is already wired to load this model when the pin in `app/config.py` is set.